# Aula 1 - Criando um agente

## Vídeo 1.2 - Entendendo a estrutura do Markdown

https://quarto.org/docs/presentations/

In [ ]:
%pip install pydantic-ai-slim[tavily]
%pip install docling
%pip install fastembed
%pip install qdrant-client

In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
import os
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

In [ ]:
from pydantic_ai import Agent
import nest_asyncio
nest_asyncio.apply()

In [ ]:
agent = Agent(
    'google-gla:gemini-2.0-flash',
    system_prompt='''Você é um criador de apresentações que cria apresentações neste formato:
    ---
title: "Habits"
author: "John Doe"
format: revealjs
---

## Getting up

- Turn off alarm
- Get out of bed

## Going to sleep

- Get in bed
- Count sheep

Ao receber um tema crie a apresentação.


    ''',
)

In [ ]:
result = agent.run_sync('Indústria de petshops')

In [ ]:
print(result.data)

## Vídeo 1.3 - Formatando resultados

https://ai.pydantic.dev/results/#result-validators-functions

In [ ]:
from pydantic import BaseModel
import yaml  # Biblioteca para manipular YAML

In [ ]:
class Formatacao(BaseModel):
    title: str
    author: str
    format: str
    theme: str
    incremental: bool


In [ ]:
agente_formatador = Agent(
    'google-gla:gemini-2.0-flash',
    system_prompt='''Você é um criador de apresentações que retorna a formatação para apresentação. A formatação tem o seguinte formato:
---
title: "Presentation"
author: "John Doe"
format:
  revealjs:
    theme: dark
    incremental: true
---

As opções de tema são:
beige
blood
dark
default
league
moon
night
serif
simple
sky
solarized

Você deve inferir quais as propriedades da formatação a partir do prompt.
''',result_type=Formatacao
)

In [ ]:
#result = agent.run_sync('Quero uma apresentação com o tema escuro, o autor é a Gatito Petshop, e o título é Ganhos em vendas de arranhadores.')
#print(result.data)

In [ ]:
result = agente_formatador.run_sync('Quero uma apresentação com o tema escuro, o autor é a Gatito Petshop, e o título é Ganhos em vendas de arranhadores.')
print(result.data)

In [ ]:
result.data.model_dump()

In [ ]:
def formatar_para_yaml(result_data):
    """
    Formata os dados de result.data no formato YAML esperado.
    """
    # Constrói a estrutura do dicionário com base nos dados fornecidos
    yaml_data = {
        "title": result_data.title,
        "author": result_data.author,
        "format": {
            result_data.format: {
                "theme": result_data.theme,
                "incremental": result_data.incremental,
            }
        },
    }

    # Converte para YAML
    import yaml
    return yaml.dump(
        yaml_data,
        sort_keys=False,  # Mantém a ordem dos campos
        default_flow_style=False,  # Gera o YAML no estilo de múltiplas linhas
    )


In [ ]:
# Formata para YAML
yaml_formatado = formatar_para_yaml(result.data)

print(yaml_formatado)

## Vídeo 1.4 - Avaliando resultados

https://ai.pydantic.dev/results/#result-validators-functions

In [ ]:
from pydantic_ai import ModelRetry

In [ ]:
@agente_formatador.result_validator
async def valida_resultado(result):
  if result.title == 'Presentation':
      raise ModelRetry(f'''
      Você precisa passar um título que tenha relação com gatos e a apresentação de resultados semestrais da Gatito Petshop.''')
  else:
    return result


In [ ]:
result = agente_formatador.run_sync('Quero uma apresentação com o tema escuro, o autor é a Gatito Petshop.')

In [ ]:
result.data

In [ ]:
yaml_formatado = formatar_para_yaml(result.data)

print(yaml_formatado)

# Aula 2 - Aplicando Agentic RAG

## Vídeo 2.1 - Carregando documentos

https://github.com/docling-project/docling

In [ ]:
url = 'https://raw.githubusercontent.com/allanspadini/curso-pydanticai/main/dados/Relatrio_Mensal_-_Gatito_Petshop.pdf'

In [ ]:
from docling.document_converter import DocumentConverter

In [ ]:
converter = DocumentConverter()
result = converter.convert(url)

In [ ]:
print(result.document.export_to_markdown())

In [ ]:
url_csv = 'https://raw.githubusercontent.com/allanspadini/curso-pydanticai/refs/heads/main/dados/comportamento_gatos_produtos.csv'

In [ ]:
result_csv = converter.convert(url_csv)

In [ ]:
print(result_csv.document.export_to_markdown())

https://ai.pydantic.dev/examples/rag/

## Vídeo 2.2 - Gerando embeddings

https://qdrant.tech/documentation/fastembed/fastembed-quickstart/

In [ ]:
from fastembed import TextEmbedding

In [ ]:
texto = result.document.export_to_markdown()

# Defina o tamanho médio de uma página (ajuste conforme necessário)
tamanho_pagina = 1800

# Divide o texto em chunks
chunks = {}
for i in range(0, len(texto), tamanho_pagina):
    chave = f"page_{(i // tamanho_pagina) + 1}"
    chunk = texto[i:i + tamanho_pagina]
    chunks[chave] = chunk.strip()

# Exibir exemplo do resultado
for k, v in chunks.items():
    print(f"\n== {k} ==\n{v[:200]}...\n")


In [ ]:
len(chunks)

In [ ]:
document = list(chunks.values()) + [result_csv.document.export_to_markdown()]

In [ ]:
embedding_model = TextEmbedding()

In [ ]:
embeddings_generator = embedding_model.embed(document)
embeddings_list = list(embeddings_generator)

In [ ]:
len(embeddings_list)

In [ ]:
embeddings_list

## Vídeo 2.3 - Buscando em uma base vetorial

In [ ]:
from qdrant_client import QdrantClient, models

#client = QdrantClient(":memory:")
client = QdrantClient(path="qdrant_db")  # Persiste no diretório qdrant_db

In [ ]:
list(chunks.keys())

In [ ]:
chunks['page_1']

In [ ]:
texto_csv = result_csv.document.export_to_markdown()
metadata = [{"source": chunks[f'page_{i+1}']} for i in range(7)]
metadata.append({"source": texto_csv})

In [ ]:
ids = list(range(8))

In [ ]:
points = [
    models.PointStruct(id=id, vector=vector, payload=payload)
    for id, (vector, payload) in zip(ids, zip(embeddings_list, metadata))
]

In [ ]:
client.create_collection(
    collection_name="relatórios",
    vectors_config={
        "size": 384,
        "distance": "Cosine"
    }
)

In [ ]:
client.upsert(
    collection_name="relatórios",
    wait=True,
    points=points
)

In [ ]:
query_embedding = embedding_model.embed("Qual a reação Mais Comum do Gato?")

In [ ]:
query_embedding=list(query_embedding)

In [ ]:
query_embedding[0]

In [ ]:
search_result = client.query_points(
    collection_name="relatórios",
    query=query_embedding[0],
    limit=1,
)
print(search_result)

In [ ]:
search_result.points[0].payload['source']

# Aula 3 - Criando ferramentas

## Vídeo 3.1 - Transformando a base em ferramenta

https://www.youtube.com/watch?v=P212vYt6Xo8

In [ ]:
from pydantic_ai.models.gemini import GeminiModel
modelo = GeminiModel('gemini-2.0-flash', provider='google-gla')

In [ ]:
agente_roteirista = Agent(
    model=modelo,
    system_prompt='''Você é um criador de apresentações que retorna o roteiro de um slide.
    A apresentação deve responder a perguntas relacionadas no título dos slides e o roteiro de cada slide
    deve ficar entre as tags ::: {.notes} roteiro ::: . O roteiro deve ser um texto corrido sem bullets e ser elaborado com
    base no resultado da consulta à ferramenta de nome consulta. Nunca escreva sem antes consultar a ferramenta consulta, mas sempre coloque algo
    nas notas para a pessoa ter o que falar no slide.

    Exemplo de resultado:

    ## título do slide

    ::: {.notes}
    roteiro do slide
    :::

'''
)

In [ ]:
perguntas = [
    "Qual o resumo executivo do relatório?",
    "Qual o faturamento com rações na vendas por categoria?",
    "Qual o faturamento com arranhadores na vendas por categoria?",
    "Qual o faturamento com brinquedos na vendas por categoria?",
    "Quais produtos de previsão de reposição urgente na análise de estoque?",
    "Como está o desempenho das vendas online comparado à loja física?",
    "Quais são os diferenciais competitivos frente à concorrência?",
    "Quais ações de RH impactaram o desempenho das equipes?",
    "Os indicadores financeitos estão alinhados com as metas?",
    "Quais tendências de consumo devem ser exploradas nos próximos meses?",
    "Qual a reação mais comum do Gato?"
]


https://ai.pydantic.dev/tools/#function-tools-and-schema

In [ ]:
@agente_roteirista.tool_plain(docstring_format='google', require_parameter_descriptions=True,retries=2)
def consulta(pergunta: str):
    """Realiza uma consulta em uma coleção de relatórios com base na pergunta fornecida.

    Args:
        pergunta (str): A pergunta ou consulta que será usada para gerar o embedding e buscar o relatório correspondente.
    Returns:
        str: O conteúdo do relatório mais relevante encontrado com base na pergunta.

    Exemplo:
        >>> consulta("Qual é o relatório mais recente sobre vendas?")
        "Relatório de vendas do trimestre Q1 2025..."
    """


    query_embedding = embedding_model.embed(pergunta)
    query_embedding = list(query_embedding)

    client = QdrantClient(path="qdrant_db")
    search_result = client.query_points(
        collection_name="relatórios",
        query=query_embedding[0],
        limit=1,
    )
    return search_result.points[0].payload['source']


In [ ]:
# Reinicie o kernel ou processo, ou use outro nome de pasta temporária
client = QdrantClient(path="qdrant_db_temp")

In [ ]:
consulta("Arranhadores de parede estão em nível crítico")

In [ ]:
perguntas[0]

In [ ]:
result = agente_roteirista.run_sync(perguntas[0])

In [ ]:
print(result.data)

In [ ]:
import time

In [ ]:
# Variável para armazenar todas as saídas
roteiro_completo = ""

# Processamento das perguntas
for pergunta in perguntas:
    time.sleep(10)  # Adicione um atraso de 1 segundo entre as perguntas
    resultado = agente_roteirista.run_sync(pergunta)
    roteiro_completo += resultado.data + "\n\n"  # Adiciona quebra de linha entre seções

In [ ]:
print(roteiro_completo)

## Vídeo 3.2 - Usando ferramentas no formato do PydanticAI

https://www.youtube.com/watch?v=MkqkiJgnDxk&t=14s

In [ ]:
from pydantic_ai.common_tools.tavily import tavily_search_tool

In [ ]:
from google.colab import userdata
tavily = userdata.get('tavily')

In [ ]:
from pydantic_ai.models.gemini import GeminiModel, GeminiModelSettings

In [ ]:
modelo = GeminiModel('gemini-2.0-flash', provider='google-gla')

In [ ]:
def funcao_buscadora():
    agente_buscador = Agent(
        model=modelo,
        tools=[tavily_search_tool(tavily)],
        system_prompt='''Você deve buscar na internet pela query e retornar uma explicação dos resultados.
        Seja sucinto na resposta, respondendo em no máximo 2 parágrafos usando texto corrido. O conteúdo da resposta deve seguir o formato:

        ## Título

        ::: {.notes }

        Resposta

        :::


    '''
    )
    result = agente_buscador.run_sync('Quais as principais fornecedores de arranhadores na internet?')
    return result

In [ ]:
resultado = funcao_buscadora()

In [ ]:
print(resultado.data)

In [ ]:
resultado.all_messages_json()

## Vídeo 3.3 - Gerando imagens

In [ ]:
from google import genai
from google.genai import types
from PIL import Image
from io import BytesIO

In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
contents = ('Hi, can you create a 3d rendered image of a cat '
            'with wings and a top hat flying over a happy '
            'futuristic scifi city with lots of greenery?')

In [ ]:
response = client.models.generate_content(
    model="gemini-2.0-flash-exp-image-generation",
    contents=contents,
    config=types.GenerateContentConfig(
      response_modalities=['Text', 'Image']
    )
)

In [ ]:
for part in response.candidates[0].content.parts:
  if part.text is not None:
    print(part.text)
  elif part.inline_data is not None:
    image = Image.open(BytesIO((part.inline_data.data)))
    image.save('gemini-native-image.png')


In [ ]:
from IPython.display import display
display(image)

In [ ]:
from typing import Literal

In [ ]:
#@agente_orquestrador.tool_plain(docstring_format='google', require_parameter_descriptions=True,retries=5)
def gera_imagem(descricao: str, n_slide: str) -> str:
    """Gera uma imagem com base na descrição fornecida.

    Args:
        descricao (str): A descrição ou prompt que será usado para gerar a imagem.
        n_slide (str): O número do slide onde a imagem será inserida.

    Returns:
        Image_path: O caminho onde a imagem foi salva

    Exemplo:
        >>> consulta("Gere uma imagem de um gato em ambiente de escritório")
        "slide1.png"
    """
    response = client.models.generate_content(
      model="gemini-2.0-flash-exp-image-generation",
      contents=descricao,
      config=types.GenerateContentConfig(
        response_modalities=['Text', 'Image']
      )
    )
    for part in response.candidates[0].content.parts:
      if part.text is not None:
        print(part.text)
      elif part.inline_data is not None:
        image = Image.open(BytesIO((part.inline_data.data)))
        image.save(n_slide+'.png')
    Image_path = n_slide+'.png'
    return Image_path

# Aula 4 - Aplicando múltiplos agentes

## Vídeo 4.1 - Completando a apresentação

In [ ]:
#@agente_orquestrador.tool_plain(docstring_format='google', require_parameter_descriptions=True,retries=5)
def funcao_agente_designer(conteudo):
      """Gera uma imagem com base na descrição fornecida.

      Args:
          conteudo (str): Conteúdo gerado pelo agente roteirista.

      Returns:
          Resultado: O conteúdo da apresentação complementado.

      Exemplo:
          >>> funcao_agente_designer(conteudo)
          "Resultado da apresentação"
      """
      agente_designer = Agent(
          model=modelo,
          system_prompt=f'''
      Você é um criador de apresentações que recebe o roteiro: {conteudo}.
      Seu trabalho é transformar esse roteiro em uma apresentação no estilo Quarto usando markdown.

      Cada slide começa com ## Título do Slide.
      Você deve:
      - Adicionar até 3 bullets, imagem ou código python
      em markdown que permita a construção de um gráfico com o que é observado nas notas.
      - Você deve escolher apenas um dos três por slide, bullets, imagens ou gráficos.
      - Manter as notas com o que será falado, usando o campo `notes:`.
      - Adicionar uma imagem em alguns dos slides. A imagem deve estar no formato markdown e estar relacionada ao número do slide.
      - Retornar a **apresentação completa** com todos os slides, incluindo aqueles com imagem gerada.

      Exemplo de chamada de imagem:
      `![Descrição de um gato que deve aparecer no slide](3.png)`

      Notas devem aparecer assim:
      ::: {{.notes}}
      Texto das notas aqui
      :::

      Não diga mais nada além do markdown final.
      '''
      )
      result = agente_designer.run_sync('Complemente a apresentação')
      Resultado = result.data
      return Resultado

In [ ]:
Resultado = funcao_agente_designer(roteiro_completo)

In [ ]:
print(Resultado)

## Vídeo 4.2 - Revisando a produção

In [ ]:
agente_revisor = Agent(
          model=modelo,
          system_prompt=f'''
          Você é o agente revisor de apresentação.
          Você deve verificar o texto passado via prompt e conferir se não existem informações como
          - [Preencha com os resultados da consulta]. Nesse caso você deve substituir esse conteúdo por
          um texto que faça sentido de acordo com as notas do slide em questão.

          Quando você observar a presença de uma imagem como por exemplo:
          ![Um gato brincando com um brinquedo](3.png)
          Você deve gerar o arquivo da imagem correspondente usando a ferramenta gera_imagem.

          Ao final você deve retornar o texto corrigido.

      '''
      )

In [ ]:
apresentacao_final = agente_revisor.run_sync(Resultado)

In [ ]:
print(apresentacao_final.data)